#**deep learning**

Deep Learning (DL) is a branch of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns from large amounts of data.


data

analysis and preprossing

neural networking

#neural network

A Neural Network is a computer system that learns from data using layers of connected nodes (neurons).

#tokenization, vectorization and embedding

**Tokenization** = breaking text into smaller pieces called tokens.

**Vectorization** = converting tokens/text into numbers.

**Embedding** = representing words/tokens as numerical vectors that capture their meaning and relationships.

#sentiment analysis system

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

#download nltk data

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

#STEP1: CREATE A DATASET

In [ ]:
data = {
    'review': [
         "I absolutely love this product! It's amazing!",
         "Best purchase ever. Highly recommend!",
         "Excellent quality and fast delivery.",
         "Very happy with this. Will buy again.",
         "Fantastic product. Works perfectly.",
         "Terrible product. Waste of money.",
         "Worst purchase ever. Very disappointed.",
         "Poor quality. Broke immediately.",
         "Bad service. Will never buy again.",
         "Horrible experience. Do not recommend.",
         "Amazing quality! Exceeded my expectations.",
         "Not worth the price. Very cheap material.",
         "Superb product! Very satisfied.",
         "Completely useless. Returned immediately.",
         "Great customer support. Very helpful.",
         "Awful product. Stopped working in a day."
        ],
        'sentiment': [
        1, 1, 1, 1, 1,  # Positive
        0, 0, 0, 0, 0,  # Negative
        1, 0, 1, 0, 1, 0
        ]
    }

In [ ]:
df=pd.DataFrame(data)
print("dataset shape:",df.shape)
print("\n first 5 rows:")
print(df.head())

In [ ]:
print(df.tail())

#STEP 2: PREPROCESSING FUNCTION

In [ ]:
lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))
def preprocess_text(text):
  text=text.lower()
  #remove punctuation
  text=text.translate(str.maketrans('','',string.punctuation))
  #remove numbers
  text=re.sub(r'\d+','',text)
  tokens=word_tokenize(text)
  tokens=[lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

  return ' '.join(tokens)

In [ ]:
print(preprocess_text("Sooooo Gunda went to the marketttt todayyy, bought five mangooooes, met Raju, slipped badly, laughed loudly, and returned home happilyyy! 😂"))

#apply preprocessing

In [ ]:
df['clean_review']=df['review'].apply(preprocess_text)
print("\npreprocessed reviews:")
print(df[['review', 'clean_review']].head())

In [ ]:
df['clean_review']=df['review'].apply(preprocess_text)
print("\npreprocessed reviews:")
print(df[['review', 'clean_review']].tail())

STEP 3:TRAIN/TEST SPLIT

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(df['clean_review'],df['sentiment'],test_size=0.2,random_state=42,stratify=df['sentiment'])


In [ ]:
len(x_train)


In [ ]:
len(x_test)

#VECTORIZATION

In [ ]:
tfidf=TfidfVectorizer(max_features=1000, ngram_range=(1,2))
x_train_tfidf=tfidf.fit_transform(x_train)
x_test_tfidf=tfidf.transform(x_test)

In [ ]:
vocab=tfidf.get_feature_names_out()
print(f"total vocabulary size: {len(vocab)} words")
print(f"\nfirst 30 words in vocabulary:")
for i in range(0, min(30, len(vocab)),5 ):
  chunk=vocab[i:i+5]
  print(f" {i:3}: {list(chunk)}")


In [ ]:
#VIEW TF-IDF VECTOR VALUES
# Convert sparse matrix into normal array for viewing
tfidf_array = x_train_tfidf.toarray()
print("TF-IDF Matrix Shape:", tfidf_array.shape)
 # Show vector values for first reviews
for i in range(min(1, len(x_train))):
 print("REVIEW", i + 1)
 print("Original Text :", x_train.iloc[i])
 print("Sentiment     :", y_train.iloc[i])
 print("\nNon-zero TF-IDF values:")
 non_zero_indices = np.where(tfidf_array[i] > 0)[0]
 for index in non_zero_indices:
  print(f"{vocab[index]:25} : {tfidf_array[i][index]:.4f}")

# STEP 4: TRAIN SENTIMENT ANALYSIS MODEL

In [ ]:
from sklearn.linear_model import LogisticRegression
# Create model
model = LogisticRegression(max_iter=1000)
# Train model
model.fit(x_train_tfidf, y_train)
print("Model training completed!")

# STEP 5: TEST THE MODEL

In [ ]:
y_pred = model.predict(x_test_tfidf)
print("Actual Sentiments:")
print(["Happy" if x == 1 else "Unhappy" for x in y_test.values])
print("\nPredicted Sentiments:")
print(["Happy" if x == 1 else "Unhappy" for x in y_pred])

# STEP 6: MODEL EVALUATION

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# STEP 7: TEST WITH YOUR OWN SENTENCE

In [ ]:
new_review = ["This product is terrible and disappointing"]
new_review_tfidf = tfidf.transform(new_review)
prediction = model.predict(new_review_tfidf)
if prediction[0] == 1:
  print("Sentiment: Happy")
else:
  print("Sentiment: Unhappy")

# STEP 8: TEST NEW REVIEWS

In [ ]:
new_reviews = ["I absolutely love this product",
               "This product is terrible and disappointing",
               "The quality is excellent",
               "The product is okay"
]
# Preprocess new reviews
clean_new_reviews = [preprocess_text(review) for review in new_reviews]
# Convert text into TF-IDF vectors
new_reviews_tfidf = tfidf.transform(clean_new_reviews)
# Predict sentiment
predictions = model.predict(new_reviews_tfidf)
print("SENTIMENT ANALYSIS RESULTS")
print("=" * 50)
for review, sentiment in zip(new_reviews, predictions):
 print(f"Review    : {review}")
 print(f"Sentiment : {sentiment}")
 print("-" * 50)

#Step 1 — Install Groq

In [ ]:
pip install -q groq

#Step 2 — Import and add your API key

In [ ]:
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-20b"
print("Groq connected successfully!")

#Step 3 — Create the chatbot

In [ ]:
while True:
  user_message = input("You: ")
  if user_message.lower() == "exit":
    print("Bot: Goodbye! 👋 ")
    break
  response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "act as a director"
            },
            {
                "role": "user",
                "content": user_message
                }
        ]

    )
  bot_reply = response.choices[0].message.content
  print("Bot:", bot_reply)